# Loops for CNN Model

# Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from model import CartoonCNN, load_checkpoint
from src.data import get_dataloaders

# Variables
## Key variables: load dataloader, initialise model, define loss function, and optimizer

In [ ]:
train_loader, val_loader, test_loader, class_names = get_dataloaders(batch_size=32)
num_classes = len(class_names)

model = CartoonCNN(num_classes=num_classes)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop

In [ ]:
num_epochs = 20
train_losses = []
best_val_loss = float('inf')  # Tracker for smallest validation loss seen so far

for epoch in range(num_epochs):

    # ------------------------------------------------------------------
    # Training phase
    # model.train() enables training-time behaviour for layers such as
    # Dropout (randomly zeros activations) and BatchNorm (uses per-batch
    # statistics). Skipping this after a validation phase would leave the
    # model stuck in eval mode and produce incorrect training dynamics.
    # ------------------------------------------------------------------
    model.train()
    for images, labels in train_loader:
        optimizer.zero_grad()                     # Clear stale gradients from the previous step
        outputs = model(images)                   # Forward pass
        loss = criterion(outputs, labels)         # Compute training loss
        loss.backward()                           # Backpropagate gradients
        optimizer.step()                          # Update weights

    # ------------------------------------------------------------------
    # Validation phase
    # model.eval() switches Dropout off and makes BatchNorm use running
    # (population) statistics instead of batch statistics, so the metrics
    # reflect true inference behaviour rather than training-time noise.
    # torch.no_grad() disables the autograd engine entirely, preventing any
    # gradients from being computed or stored during validation — this saves
    # memory and guarantees that val data never influences weight updates.
    # ------------------------------------------------------------------
    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            outputs = model(images)
            total_val_loss += criterion(outputs, labels).item()

    # Average loss over every validation batch.
    # The original code only kept the last batch's loss, which is a biased
    # and unreliable signal — the 'best' checkpoint could be selected based
    # on a single easy batch rather than overall validation performance.
    avg_val_loss = total_val_loss / len(val_loader)

    # Restore training mode so the next epoch's forward passes are correct.
    model.train()

    # ------------------------------------------------------------------
    # Checkpoint: persist the model only when val loss genuinely improves.
    # ------------------------------------------------------------------
    if avg_val_loss < best_val_loss:
        print(f"  Validation loss improved from {best_val_loss:.4f} to {avg_val_loss:.4f} — saving model...")
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_model.pth')

    train_losses.append(loss.item())
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {loss:.4f} | Val Loss: {avg_val_loss:.4f}")
